# 4.0 Benchmark (run in SageMaker Studio)

Simple baseline for Part C: a depth-3 decision tree (scikit-learn, trained locally) on two features - chest acc16 magnitude std and HR mean (NaN filled with train median). A majority-class row is reported as the floor. Primary metric: macro-F1 on test. Test scores are optimistic (seen subjects, per spec B).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json
import boto3, numpy as np, pandas as pd
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_parquet(name):
    b, k = split_key(SPLITS + name)
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
def read_json(uri):
    b, k = split_key(uri)
    return json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
CLASSES = list(range(K))
def class_name(i):
    return ACTIVITY_NAMES[LABEL_MAP[int(i)]]
print(K, "classes;", len(FEATURES), "features")


## Load splits and fit

In [ ]:
from src.pamap2.benchmark import BENCHMARK_FEATURES, fit_benchmark, predict_benchmark, predict_majority
train, val, test = (read_parquet(f"{n}.parquet") for n in ("train", "validation", "test"))
for df in (train, val, test):
    E.check_feature_columns(df, FEATURES)
model = fit_benchmark(train)
print("Features:", BENCHMARK_FEATURES, "| train medians:", model["medians"], "| majority class:", class_name(model["majority"]))

## Metrics on validation and test

In [ ]:
report = {"features": BENCHMARK_FEATURES, "model": "DecisionTree(max_depth=3)",
          "label_map": {str(k): v for k, v in LABEL_MAP.items()}}
for name, df in (("validation", val), ("test", test)):
    for label, pred in (("tree", predict_benchmark(model, df)), ("majority", predict_majority(model, df))):
        report[f"{name}_{label}"] = E.classification_metrics(df["label"], pred, CLASSES)
summary = pd.DataFrame([{"split": s, "model": m, "macro_f1": report[f"{s}_{m}"]["macro_f1"],
                         "accuracy": report[f"{s}_{m}"]["accuracy"]}
                        for s in ("validation", "test") for m in ("tree", "majority")])
display(summary)

In [ ]:
pc = report["test_tree"]["per_class"]
display(pd.DataFrame({class_name(k): v for k, v in pc.items()}).T)

## Save report

In [ ]:
write_json(REPORTS + "benchmark.json", report)